# 03 — Generate the real-Argon REFPROP Seitz surface

This notebook is independent of the MD cavitation calculation.  It evaluates actual
Argon over a controlled temperature-pressure grid.  Pressure is parameterized as a
fraction of the saturation pressure at each temperature, which keeps the grid in the
superheated-liquid region.

Failed REFPROP/Seitz points remain visible in the output table and are excluded from
plots rather than silently interpolated.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

MD_REPO = Path.home() / "MDsims"
DATA_DIRECTORY = MD_REPO / "Argon" / "data"
SEITZ_REPO = Path.home() / "SeitzModel"
REFPROP_ROOT = SEITZ_REPO / "REFPROP"
REFPROP_LIBRARY = REFPROP_ROOT / "lib" / "librefprop.so.2.30"

if str(REFPROP_ROOT) not in sys.path:
    sys.path.insert(0, str(REFPROP_ROOT))

from ctREFPROP.ctREFPROP import REFPROPFunctionLibrary
from SeitzModel import SeitzModel

rp = REFPROPFunctionLibrary(str(REFPROP_LIBRARY))
rp.SETPATHdll(str(REFPROP_ROOT))
ierr, herr = rp.SETUPdll(
    1,
    str(REFPROP_ROOT / "FLUIDS" / "ARGON.FLD"),
    str(REFPROP_ROOT / "FLUIDS" / "HMX.BNC"),
    "DEF",
)
assert ierr == 0, herr
print("REFPROP version:", rp.RPVersion())


## Define and inspect the physical grid


In [ ]:
temperatures_K = np.arange(90.0, 141.0, 5.0)
pressure_fractions = np.array([0.10, 0.25, 0.40, 0.55, 0.70, 0.85])

saturation_rows = []
for temperature_K in temperatures_K:
    sat = rp.SATTdll(float(temperature_K), [1.0], 1)
    saturation_rows.append({
        "temperature_K": temperature_K,
        "Pvap_kPa": sat.P,
        "Pvap_bar": sat.P / 100.0,
        "rho_liquid_mol_L": sat.Dl,
        "rho_vapor_mol_L": sat.Dv,
        "ierr": sat.ierr,
        "message": sat.herr,
    })
saturation_table = pd.DataFrame(saturation_rows)
saturation_table


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
axes[0].plot(saturation_table["temperature_K"], saturation_table["Pvap_bar"], "o-")
axes[0].set(xlabel="Temperature (K)", ylabel="Saturation pressure (bar)", title="Argon saturation boundary")
axes[0].grid(alpha=0.3)
axes[1].plot(saturation_table["temperature_K"], saturation_table["rho_liquid_mol_L"], "o-", label="Liquid")
axes[1].plot(saturation_table["temperature_K"], saturation_table["rho_vapor_mol_L"], "s-", label="Vapor")
axes[1].set(xlabel="Temperature (K)", ylabel="Saturation density (mol/L)", title="Saturation densities", yscale="log")
axes[1].legend()
axes[1].grid(alpha=0.3, which="both")
plt.show()


In [ ]:
grid_rows = []
for row in saturation_table.itertuples(index=False):
    for fraction in pressure_fractions:
        pressure_bar = fraction * row.Pvap_bar
        grid_rows.append({
            "temperature_K": row.temperature_K,
            "temperature_C": row.temperature_K - 273.15,
            "pressure_fraction_of_Pvap": fraction,
            "pressure_bar": pressure_bar,
            "pressure_psia": pressure_bar * 14.5037738,
            "Pvap_bar_direct": row.Pvap_bar,
        })
grid = pd.DataFrame(grid_rows)
print("Grid points:", len(grid))
grid


## Evaluate SeitzModel at every paired temperature-pressure point


In [ ]:
reference = SeitzModel(
    grid["pressure_psia"].to_numpy(dtype=float),
    grid["temperature_C"].to_numpy(dtype=float),
    "ARGON",
    [1.0],
)

fields = {
    "Q_keV": reference.Q,
    "Rc_nm": reference.Rc,
    "Pvap_psia": reference.Pvap,
    "Pbub_psia": reference.Pbub,
    "Rho_l_g_cc": reference.Rho_l,
    "Rho_b_g_cc": reference.Rho_b,
    "Sigma_N_m": reference.Sigma,
    "P_err_psia": reference.P_err,
    "G_err_J_kg": reference.G_err,
    "iterations": reference.interp_iterations,
    "errID": reference.errID,
}
for name, values in fields.items():
    grid[name] = np.asarray(values, dtype=object)

numeric = [name for name in fields if name != "errID"]
grid[numeric] = grid[numeric].apply(pd.to_numeric, errors="coerce")
grid["valid"] = (
    grid["errID"].isna()
    & grid["Q_keV"].gt(0)
    & grid["Rc_nm"].gt(0)
    & grid["Pvap_psia"].gt(grid["pressure_psia"])
    & grid["Pbub_psia"].gt(grid["pressure_psia"])
)

print("Valid points:", int(grid["valid"].sum()), "/", len(grid))
grid


## Visualize the real-Argon Seitz surface


In [ ]:
valid = grid.loc[grid["valid"]].copy()
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5), constrained_layout=True)
for temperature, group in valid.groupby("temperature_K"):
    group = group.sort_values("pressure_fraction_of_Pvap")
    axes[0].plot(group["pressure_fraction_of_Pvap"], group["Q_keV"], "o-", label=f"{temperature:g} K")
    axes[1].plot(group["pressure_fraction_of_Pvap"], group["Rc_nm"], "o-", label=f"{temperature:g} K")
axes[0].set(xlabel="P / Pvap", ylabel="REFPROP Seitz Q (keV)", title="Real-Argon threshold", yscale="log")
axes[1].set(xlabel="P / Pvap", ylabel="Critical radius (nm)", title="Real-Argon critical radius", yscale="log")
for axis in axes:
    axis.legend(ncol=2, fontsize=8)
    axis.grid(alpha=0.3, which="both")
plt.show()


In [ ]:
q_surface = valid.pivot(index="temperature_K", columns="pressure_fraction_of_Pvap", values="Q_keV")
rc_surface = valid.pivot(index="temperature_K", columns="pressure_fraction_of_Pvap", values="Rc_nm")

fig, axes = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)
for axis, surface, title, label in [
    (axes[0], q_surface, "Q surface", "Q (keV)"),
    (axes[1], rc_surface, "Critical-radius surface", "Rc (nm)"),
]:
    image = axis.imshow(surface.to_numpy(), origin="lower", aspect="auto",
                        extent=[surface.columns.min(), surface.columns.max(),
                                surface.index.min(), surface.index.max()])
    axis.set(xlabel="P / Pvap", ylabel="Temperature (K)", title=title)
    fig.colorbar(image, ax=axis, label=label)
plt.show()


In [ ]:
output_path = DATA_DIRECTORY / "argon_refprop_seitz_grid.csv"
grid.to_csv(output_path, index=False)
print("Saved:", output_path)
